In [1]:
library(tidyverse)
library(DESeq2)
library(BiocParallel)

Warning message in system("timedatectl", intern = TRUE):
“running command 'timedatectl' had status 1”
── Attaching packages ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.5     ✔ purrr   0.3.4
✔ tibble  3.1.5     ✔ dplyr   1.0.7
✔ tidyr   1.1.4     ✔ stringr 1.4.0
✔ readr   2.0.2     ✔ forcats 0.5.1

── Conflicts ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: parallel


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:parallel’:

    clusterApply, clusterApplyLB, clusterCall, clusterEvalQ,
    clusterExport, clusterMap, parApply, parCapply

In [2]:
# Set number of cores to use
n_cores <- detectCores() - 2
BiocParallel::register(MulticoreParam(n_cores))

In [3]:
counts_df <- read_tsv("/mnt/d/gene_expression_example_data/rna_seq_ucec_counts.tsv")
coldata_df <- read_tsv("/mnt/d/gene_expression_example_data/rna_seq_ucec_coldata.tsv")
counts_df <- counts_df %>%
    select(-Entrez_Gene_Id) %>%
    dplyr::rename(symbol = Hugo_Symbol) %>%
    mutate_if(is.numeric, round, 0)
coldata_df <- coldata_df %>%
    mutate(condition = factor(condition, levels = c("healthy", "tumor")))

Rows: 20242 Columns: 248

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr   (1): Hugo_Symbol
dbl (247): Entrez_Gene_Id, GTEX-T6MO-1526-SM-4DM57, GTEX-11P81-1626-SM-5BC52...


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

Rows: 246 Columns: 3

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): sample_name, condition, data_source


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.



In [4]:
min_expr <- 0
min_prop <- 1/3
padj_thresh <- 0.05

In [5]:
filt_counts_df <- counts_df %>%
    filter(rowSums(.[-1] > min_expr) / (ncol(.) - 1) >= min_prop)
filt_expr <- filt_counts_df %>%
    column_to_rownames("symbol") %>%
    as.matrix()

In [6]:
filt_expr

,GTEX-T6MO-1526-SM-4DM57,GTEX-11P81-1626-SM-5BC52,GTEX-13N11-1126-SM-5KM41,GTEX-RTLS-2426-SM-46MUO,GTEX-ZP4G-0726-SM-4WWF2,GTEX-WEY5-0726-SM-4LMID,GTEX-T2IS-2226-SM-4DM65,GTEX-PX3G-2026-SM-48U1H,GTEX-U3ZN-0726-SM-4DXT5,GTEX-XUW1-0226-SM-4BOOS,⋯,TCGA-EO-A3AU-01A-21R-A19W-07,TCGA-AJ-A3EK-01A-11R-A19W-07,TCGA-5S-A9Q8-01A-11R-A40A-07,TCGA-BK-A139-01A-11R-A277-07,TCGA-D1-A3DG-01A-11R-A19W-07,TCGA-EO-A22X-01A-11R-A180-07,TCGA-KP-A3W1-01A-11R-A22K-07,TCGA-PG-A916-01A-11R-A37O-07,TCGA-EY-A4KR-01A-11R-A27V-07,TCGA-KP-A3VZ-01A-11R-A22K-07
FAM208A,2524,2703,2036,2205,3770,2313,1588,2703,1974,1861,⋯,2437,3907,836,7408,4173,1710,2587,2609,2578,3914
RADIL,320,183,209,577,384,199,286,345,360,341,⋯,7,5,2,1,968,35,9,322,30,344
AP1M2,34,8,24,2,36,12,7,9,33,10,⋯,5292,7638,1663,3209,7544,4984,6025,8132,11969,15796
TAF1,1776,2509,1957,2632,3631,2132,1417,3048,1752,1866,⋯,1849,1886,465,2886,1292,1708,1709,2215,1455,2218
SIGLEC5,4,16,31,6,50,34,6,3,4,9,⋯,11,48,2,94,36,39,4,18,13,2
KLF1,0,0,1,1,0,1,0,0,1,0,⋯,1,9,2,10,12,10,9,24,12,42
USHBP1,440,430,1165,723,727,504,589,952,399,602,⋯,62,51,28,64,88,145,138,912,120,135
LRRC38,0,4,6,0,0,8,8,2,2,0,⋯,1,0,0,0,0,4,1,0,3,18
NKPD1,7,6,4,18,9,5,2,21,12,7,⋯,83,1,22,34,32,8,10,9,42,26
SLC26A8,5,13,5,0,7,1,4,7,13,4,⋯,4,1,8,19,0,1,21,22,6,6


In [7]:
coldata <- coldata_df %>%
    column_to_rownames("sample_name")

In [8]:
coldata

,condition,data_source
,<fct>,<chr>
GTEX-T6MO-1526-SM-4DM57,healthy,GTEx
GTEX-11P81-1626-SM-5BC52,healthy,GTEx
GTEX-13N11-1126-SM-5KM41,healthy,GTEx
GTEX-RTLS-2426-SM-46MUO,healthy,GTEx
GTEX-ZP4G-0726-SM-4WWF2,healthy,GTEx
GTEX-WEY5-0726-SM-4LMID,healthy,GTEx
GTEX-T2IS-2226-SM-4DM65,healthy,GTEx
GTEX-PX3G-2026-SM-48U1H,healthy,GTEx
GTEX-U3ZN-0726-SM-4DXT5,healthy,GTEx


In [9]:
dds <- DESeqDataSetFromMatrix(
    countData = filt_expr,
    colData = coldata,
    design = ~ condition
)

converting counts to integer mode



In [11]:
dds_seq <- DESeq(dds, parallel = TRUE)
fit_res <- results(
    dds_seq,
    contrast = c("condition", "tumor", "healthy"),
    pAdjustMethod = "BH",
    alpha = padj_thresh,
    parallel = TRUE
)


estimating size factors

estimating dispersions

gene-wise dispersion estimates: 14 workers

mean-dispersion relationship

final dispersion estimates, fitting model and testing: 14 workers

-- replacing outliers and refitting for 1856 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

estimating dispersions

fitting model and testing



In [12]:
dge_res_df <- as_tibble(fit_res, rownames = "symbol")
dge_res_df

symbol,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
FAM208A,3043.622526,-0.22655764,0.05217910,-4.3419229,1.412411e-05,1.968457e-05
RADIL,309.665544,-2.24451642,0.17564923,-12.7784020,2.164716e-37,7.972888e-37
AP1M2,3378.724704,4.21079197,0.25595744,16.4511410,8.230311e-61,5.469690e-60
TAF1,2244.948024,-0.74801492,0.06089201,-12.2842878,1.100095e-34,3.769410e-34
SIGLEC5,24.745319,0.59224018,0.20044839,2.9545769,3.130981e-03,3.879090e-03
KLF1,5.088558,2.74387852,0.21218502,12.9315375,2.987833e-38,1.123487e-37
USHBP1,452.345308,-2.27625433,0.13681376,-16.6376124,3.721644e-62,2.562496e-61
LRRC38,5.067697,-0.32169426,0.32802903,-0.9806884,3.267464e-01,3.490027e-01
NKPD1,32.991438,1.68750453,0.16305123,10.3495360,4.205187e-25,1.106201e-24
